# Day 12 + 13 · Ingestion and Streaming
## Building something that keeps working after you go home

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

You are a data engineer at a payments company. It is your first month.

| | |
|---|---|
| A file lands in a folder at 2 AM | nobody is awake |
| By 6 AM the dashboard is already correct | nobody pressed anything |

Somebody built that once. This afternoon you build it.

## Configuration

Change `MY_ID` to your own name or roll number, then run this cell.
Everything you build today lives under that name, so nobody overwrites anybody.

In [ ]:
import re
from pyspark.sql import functions as F

MY_ID = "change_me"           # <-- put your name or roll number here

if MY_ID == "change_me":
    raise ValueError("Set MY_ID to your own name or roll number, then run this cell again.")

TAG      = re.sub(r"[^a-z0-9]+", "_", MY_ID.lower()).strip("_")
CATALOG  = "workspace"
SCHEMA   = f"day1213_{TAG}"

VOLUME   = "landing"
LANDING  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/incoming"
CHK_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoints"

BRONZE   = f"{CATALOG}.{SCHEMA}.payments_bronze"
COPYTGT  = f"{CATALOG}.{SCHEMA}.payments_copyinto"
CLEAN    = f"{CATALOG}.{SCHEMA}.payments_clean"
SILVER   = f"{CATALOG}.{SCHEMA}.city_totals"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

dbutils.fs.mkdirs(LANDING)


def show_files(path):
    """List a folder safely — an empty folder is a normal answer, not an error."""
    files = dbutils.fs.ls(path)
    for f in sorted(files, key=lambda x: x.name):
        print(f"  {f.name:<34}{f.size:>9} bytes")
    print(f"  ({len(files)} file(s))" if files else "  (folder is empty)")


print(f"schema   : {CATALOG}.{SCHEMA}")
print(f"landing  : {LANDING}")
print(f"bronze   : {BRONZE}")

---
# 1 · Where data lands

Before anything can be processed it has to **arrive somewhere**. In every company that
place has a name and a rule about who may write to it.

On Databricks that place is a **Unity Catalog Volume** — a folder the catalog knows about.
It knows who owns it, who may read it, and what is inside. A random folder on a machine
knows none of those things.

The cell above already created one. It is empty right now.

In [ ]:
show_files(LANDING)

### A file arrives

In real life an upstream system drops the file. Today we write our own, so that a file
can "arrive" whenever we want one.

Every file is **100 rows** of payment transactions. Keep that number in mind — it is how
you will read every result for the rest of the session.

In [ ]:
import random

CITIES = ["Hyderabad", "Mumbai", "Bengaluru", "Pune", "Delhi", "Chennai"]

def make_csv(batch, n_rows=100):
    """Build the text of one CSV file. Same batch number always gives the same rows."""
    rnd = random.Random(batch)
    lines = ["txn_id,city,amount,status,txn_time"]
    for i in range(n_rows):
        lines.append("{},{},{},{},{}".format(
            f"T{batch:02d}{i:04d}",
            rnd.choice(CITIES),
            rnd.randrange(50, 5000),
            rnd.choice(["SUCCESS", "SUCCESS", "SUCCESS", "FAILED"]),
            f"2026-08-23 {rnd.randrange(0, 24):02d}:{rnd.randrange(0, 60):02d}:00"))
    return "\n".join(lines) + "\n"


def drop_file(batch, n_rows=100, name=None):
    """Simulate one file arriving in the landing folder."""
    name = name or f"payments_batch_{batch:02d}.csv"
    path = f"{LANDING}/{name}"
    body = make_csv(batch, n_rows)
    try:
        with open(path, "w") as fh:
            fh.write(body)
    except Exception:
        dbutils.fs.put(path, body, True)
    print(f"arrived : {name}   rows: {n_rows}")
    return name


for b in (1, 2, 3):
    drop_file(b)

In [ ]:
show_files(LANDING)

### The obvious way to read them

Point Spark at the folder. Not at a file — at the **folder**. Spark reads everything in it.

In [ ]:
naive = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(LANDING))

print("rows read:", naive.count())
display(naive.limit(10))

Three files of 100 rows. Three hundred rows. Nothing surprising yet.

Spark can also tell you which file each row came from. This becomes useful the moment
something goes wrong at 2 AM and somebody asks *which file*.

In [ ]:
display(
    spark.read.option("header", True).csv(LANDING)
         .select(F.col("_metadata.file_name").alias("came_from"))
         .groupBy("came_from").count()
         .orderBy("came_from")
)

---
# 2 · Two problems nobody mentions on day one

### Problem one — the same file arrives twice

Upstream systems resend. Somebody re-uploads. It happens constantly.

In [ ]:
drop_file(1, name="payments_batch_01_RESENT.csv")

again = spark.read.option("header", True).csv(LANDING)

print("rows now         :", again.count())
print("distinct txn_ids :", again.select("txn_id").distinct().count())

Four hundred rows, three hundred transactions. One hundred of them are counted twice.
Every total built on this is now wrong, and nothing errored.

Remove the resent file and the numbers agree again.

In [ ]:
dbutils.fs.rm(f"{LANDING}/payments_batch_01_RESENT.csv")

check = spark.read.option("header", True).csv(LANDING)
print("rows now         :", check.count())
print("distinct txn_ids :", check.select("txn_id").distinct().count())

### Problem two — the one that kills the job slowly

Files keep arriving. A fourth one lands.

In [ ]:
drop_file(4)

print("rows read:", spark.read.option("header", True).csv(LANDING).count())

Four hundred rows. But only **one hundred** of them are new.

Spark just read the three files it had already read yesterday. Tomorrow it will read four.
Next month it will read three hundred. The job gets slower every single day until one
morning it does not finish before the business opens.

| | Reading the whole folder |
|---|---|
| Duplicate file arrives | rows are silently double counted |
| Nothing changes at all | the full cost is paid again |

So the tool has to **remember what it already read**.

---
# 3 · Auto Loader — the folder that remembers

Auto Loader reads the same folder, but it keeps a note of every file it has finished.
That note lives in a folder called a **checkpoint**.

Three options do the work. Read them as three questions:

| Option | The question it answers |
|---|---|
| `cloudFiles.format` | what kind of file am I reading? |
| `cloudFiles.schemaLocation` | where do I remember the columns I found? |
| `checkpointLocation` | where do I remember the files I already read? |

Run the cell below. Then run it a second time without changing anything.

In [ ]:
CHK_BRONZE = f"{CHK_ROOT}/bronze"

before = spark.table(BRONZE).count() if spark.catalog.tableExists(BRONZE) else 0

query = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "csv")
              .option("cloudFiles.schemaLocation", f"{CHK_BRONZE}/schema")
              .option("cloudFiles.inferColumnTypes", True)
              .option("header", True)
              .load(LANDING)
         .writeStream
              .option("checkpointLocation", CHK_BRONZE)
              .trigger(availableNow=True)
              .toTable(BRONZE))

query.awaitTermination()

after = spark.table(BRONZE).count()
print(f"rows before : {before}")
print(f"rows after  : {after}")
print(f"new rows    : {after - before}")

### Run it again

Go back to the cell above and run it a second time. No files were added in between.

`new rows` comes back as **0**.

That is the most important nothing in this notebook. Auto Loader looked in the folder,
compared it against its checkpoint, found no file it had not already finished, and did no
work at all.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {BRONZE}").select("version", "operation", "operationMetrics"))

### Now a real new file arrives

In [ ]:
drop_file(5)

show_files(LANDING)

Run the Auto Loader cell once more. `new rows` will be **100** — the fifth file, and
nothing else. That is incremental ingestion, and it is the whole of Day 12.

---

### Two things the checkpoint is not

| It is not | Because |
|---|---|
| a copy of your data | it stores **what has already been done**, not the rows |
| a duplicate-row detector | it works at **file** level — the same rows in a *differently named* file are new files to it |

That second row matters. Auto Loader solved problem two. Problem one — the resent file —
is still yours to solve, with `MERGE` on a business key. You already know how to do that
from the last session.

### One extra column you did not ask for

Look at the bronze table and you will find a column called `_rescued_data`. Auto Loader
adds it on purpose: if a future file has a value that does not fit the columns it knows
about, the value is parked there instead of being thrown away. It is normally empty, and
the day it is not empty is the day it saves you.

### What is actually in the checkpoint folder

In [ ]:
show_files(CHK_BRONZE)

---
## YOUR TURN · 1

1. Drop **two** files at once: `drop_file(6)` and `drop_file(7)`.
2. Before you run Auto Loader, write down the number you expect `new rows` to print.
3. Run it. Then explain to the person next to you why the number is what it is.

In [ ]:
# YOUR CODE HERE
# drop_file(6)
# drop_file(7)
# ... then re-run the Auto Loader cell above

---
# 4 · COPY INTO — the simpler tool

`COPY INTO` also loads a folder and also skips files it has already loaded. It keeps its
record inside the target table's own history rather than in a checkpoint folder.

No streaming, no checkpoint to manage. Run it twice and watch the same thing happen.

In [ ]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {COPYTGT} (txn_id STRING, city STRING, amount INT, status STRING, txn_time STRING)")

spark.sql(f"""
COPY INTO {COPYTGT}
FROM '{LANDING}'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'true')
""")

print("rows in copy target:", spark.table(COPYTGT).count())

Run that cell again. The row count does not move — every file in the folder is already
recorded against this table.

### Which one do you use?

| Situation | Tool | Why |
|---|---|---|
| One large file, loaded once | `COPY INTO` | nothing to remember, simple wins |
| Thousands of small files, all day | Auto Loader | finding the new files *is* the cost |
| A monthly reload of the same files | `COPY INTO` | easy to re-run by hand |
| Partner feed whose columns drift | Auto Loader | `schemaLocation` handles the drift |

Both are correct answers. Auto Loader is what you build a job around. `COPY INTO` is what
you run when somebody sends you a file.

---
# 5 · You have already written a streaming job

Look again at the Auto Loader cell. It says `spark.readStream` and `.writeStream`.

That was Structured Streaming. You did not notice, because it looks exactly like the
batch code you have been writing since Day 6.

| Batch | Streaming |
|---|---|
| `spark.read` | `spark.readStream` |
| `df.write` | `df.writeStream` |
| `.filter`, `.select`, `.groupBy` | exactly the same |
| Catalyst plans it | the same Catalyst plans it |

Streaming is not a different Spark. It is the same Spark, told not to stop.

### A stream that reads a Delta table

A Delta table can be a streaming **source** as well as a sink. Here we stream out of
bronze, keep only the successful payments, and append them to a clean table.

Notice it needs its own checkpoint. One query, one checkpoint. Always.

In [ ]:
CHK_CLEAN = f"{CHK_ROOT}/clean"

clean_q = (spark.readStream
                .table(BRONZE)
                .where("status = 'SUCCESS'")
           .writeStream
                .option("checkpointLocation", CHK_CLEAN)
                .trigger(availableNow=True)
                .toTable(CLEAN))

clean_q.awaitTermination()

print("bronze rows :", spark.table(BRONZE).count())
print("clean rows  :", spark.table(CLEAN).count())

Clean is smaller than bronze, because the failed payments were left behind. Run the cell
again — nothing moves, for the same reason as before.

### The one genuinely new idea — the trigger

| Trigger | What it does | When you use it |
|---|---|---|
| `availableNow=True` | process everything waiting, then **stop** | scheduled jobs — what we use today |
| `processingTime='30 seconds'` | wake up every 30 seconds, forever | near-real-time dashboards |
| *(default)* | run continuously, as fast as it can | rare, and expensive |

Everything in this notebook uses `availableNow`, which stops on its own. Always-on
streaming is restricted on the free tier and consumes your quota while you are not
looking, so do not start one at home to see what happens.

### And one idea for later

A **watermark** is how long Spark agrees to wait for late data before it closes the books
on a time window. You need it the day you start aggregating by event time. That day is
not today.

---
## YOUR TURN · 2

Build a second stream out of `BRONZE` that keeps only payments above 1000 rupees and
writes them to a table called `payments_large`.

It needs its **own** checkpoint folder. Use `f"{CHK_ROOT}/large"`.

In [ ]:
# YOUR CODE HERE
# LARGE     = f"{CATALOG}.{SCHEMA}.payments_large"
# CHK_LARGE = f"{CHK_ROOT}/large"
#
# q = (spark.readStream
#           ...
#      .writeStream
#           ...
#           .toTable(LARGE))
# q.awaitTermination()

---
# 6 · End to end — one number on a screen

Files arrive. Bronze grows by itself. Clean follows bronze. Now the last step: the
number a human actually looks at.

In [ ]:
city_totals = (spark.read.table(CLEAN)
                    .groupBy("city")
                    .agg(F.sum("amount").alias("total_amount"),
                         F.count("*").alias("txns")))

city_totals.write.mode("overwrite").saveAsTable(SILVER)

display(spark.table(SILVER).orderBy(F.desc("total_amount")))

In [ ]:
display(spark.sql(f"SELECT sum(total_amount) AS total_collected FROM {SILVER}"))

### Now make it move

Run the next cell. Then re-run, in order:

1. the **Auto Loader** cell (section 3)
2. the **clean stream** cell (section 5)
3. the two cells just above

The number changes. You dropped a file and touched nothing else.

In [ ]:
drop_file(8)

---
## YOUR TURN · 3

1. Drop file 9.
2. Write down what you expect `total_collected` to do — go up, go down, or stay the same.
3. Run the three cells in order and find out.
4. In one line: what would break if you deleted `CHK_BRONZE` and ran everything again?

In [ ]:
# YOUR CODE HERE

---
# What you built this afternoon

| Layer | What it is | How it stays current |
|---|---|---|
| Landing volume | files arriving from outside | somebody else's job |
| Bronze table | every row, exactly once per file | Auto Loader + checkpoint |
| Clean table | successful payments only | a stream reading bronze |
| City totals | the number on the screen | rebuilt from clean |

This morning you could read a file. Now you can build something that keeps reading files
after you have gone home, without ever doing the same work twice.

The one thing still missing: somebody has to press the cells. Next session that job goes
away too.

---
## Homework

1. Drop two more files, run the pipeline, and note the row count before and after.
2. In two lines: what would break if you deleted your checkpoint folder?
3. Bring your `total_collected` number to the next session.

---
## Clean up (optional)

Run this only when you are finished. It removes everything you created today.

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")